# Practical 12: Generative Adversarial Network (GAN)

**Problem Statement:** Develop a GAN model to generate synthetic handwritten digit images.

**Activities:**
1. Implement Generator and Discriminator
2. Train GAN model
3. Visualize generated images

**Dataset:** MNIST — 70,000 grayscale images of handwritten digits (0-9), 28x28 pixels.

**Note:** Enable a GPU runtime (Runtime -> Change runtime type -> GPU) before running this notebook.

## 1. Import Libraries and Load Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, _), (_, _) = keras.datasets.mnist.load_data()

X_train = X_train.astype('float32')
X_train = (X_train - 127.5) / 127.5
X_train = X_train.reshape(-1, 28, 28, 1)

BUFFER_SIZE = X_train.shape[0]
BATCH_SIZE = 256
NOISE_DIM = 100

train_dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
print("Train shape:", X_train.shape)

## 2. Implement the Generator

The generator maps a random noise vector to a 28x28 image using transposed convolutions, progressively upsampling from a small spatial size to full image resolution.

In [ ]:
def build_generator():
    model = keras.Sequential([
        keras.layers.Input(shape=(NOISE_DIM,)),
        keras.layers.Dense(7 * 7 * 128, use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(),
        keras.layers.Reshape((7, 7, 128)),

        keras.layers.Conv2DTranspose(64, 5, strides=1, padding='same', use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(),

        keras.layers.Conv2DTranspose(32, 5, strides=2, padding='same', use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(),

        keras.layers.Conv2DTranspose(1, 5, strides=2, padding='same', use_bias=False, activation='tanh')
    ])
    return model

generator = build_generator()
generator.summary()

## 3. Implement the Discriminator

The discriminator is a CNN binary classifier that estimates whether an input image is real (from MNIST) or fake (produced by the generator).

In [ ]:
def build_discriminator():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(64, 5, strides=2, padding='same'),
        keras.layers.LeakyReLU(),
        keras.layers.Dropout(0.3),

        keras.layers.Conv2D(128, 5, strides=2, padding='same'),
        keras.layers.LeakyReLU(),
        keras.layers.Dropout(0.3),

        keras.layers.Flatten(),
        keras.layers.Dense(1)
    ])
    return model

discriminator = build_discriminator()
discriminator.summary()

## 4. Define Losses and Optimizers

The discriminator is trained to correctly separate real from fake images, while the generator is trained to produce images the discriminator classifies as real.

In [ ]:
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = keras.optimizers.Adam(1e-4)
discriminator_optimizer = keras.optimizers.Adam(1e-4)

## 5. Train the GAN

Each training step updates the discriminator and generator together: the discriminator sees a batch of real images and a batch of generator-produced images, and the generator is updated based on how well it fooled the discriminator.

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([images.shape[0], NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gen_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gen_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_gradients, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
def generate_and_plot(model, epoch, seed):
    predictions = model(seed, training=False)
    fig = plt.figure(figsize=(6, 6))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    plt.suptitle(f'Epoch {epoch}')
    plt.show()

EPOCHS = 40
seed = tf.random.normal([16, NOISE_DIM])

for epoch in range(1, EPOCHS + 1):
    for batch in train_dataset:
        gen_loss, disc_loss = train_step(batch)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}: Generator Loss = {gen_loss:.4f}, Discriminator Loss = {disc_loss:.4f}")
        generate_and_plot(generator, epoch, seed)

## 6. Visualize Final Generated Images

In [ ]:
final_seed = tf.random.normal([16, NOISE_DIM])
generate_and_plot(generator, EPOCHS, final_seed)

## Conclusion

In this practical, we:
- Implemented a convolutional Generator and Discriminator
- Trained them adversarially using a custom training loop
- Visualized synthetic handwritten digit images produced by the Generator across training